## Executive Summary

**Purpose:** Go from "ratio 3, 40 N·m peak on a 15 mm input shaft" to a checked single-stage helical pair and a manufacturing package, in one call.

**What it does:** `HelicalRequirements` states what the stage must do (ratio, torques, speed, life, shafts, design priority). `design_helical_pair` searches helix angles, tooth counts, modules and face widths. It builds and checks each candidate (`HelicalPair`), rates it for fatigue at the rated torque and statically at the peak torque, and sizes its hubs, shafts and bearings. It returns the passing designs, smallest first. `HelicalDesign.write_package` writes the report, the gear data tables, the parts list, the component dimensions, a layout drawing (SVG) and DXF outlines of both gears.

**Helix angle as a judgement:** the search tries the helix angles of the chosen **priority** (`'quiet'`, `'balanced'` or `'low_thrust'`, see `gears.helical.layout`). The report states the reason for the angle it lands on, with the overlap ratio and the axial force it costs.

## How It Fits Into the Helical Gear Workflow

**Upstream:** every other `gears.helical` module.

**Downstream:** the package files. `gears.helical` re-exports the whole API.

In [ ]:
#| default_exp gears.helical.design

In [ ]:
#| export
from __future__ import annotations

import csv
import math
import os
from dataclasses import dataclass, field, asdict

import numpy as np

from emachines.gears.involute import standard_modules
from emachines.gears.rating import MATERIALS, GearMaterial
from emachines.gears.manufacturing import DataRow, write_dxf, fmt_fit, it_grade, iso286_limits
from emachines.gears.helical.geometry import helix_for_centre_distance, helical_shift_sum
from emachines.gears.helical.layout import (HelicalPair, pair_tooth_counts, helix_angles, face_width_for_overlap,
                                            split_shift_sum, HELIX_PRIORITIES, HELIX_REASONS)
from emachines.gears.helical.rating import HelicalRatingInputs, HelicalPairRating, rate_helical_pair
from emachines.gears.helical.components import (PairComponentInputs, PairComponents, pair_rims,
                                                size_pair_components)
from emachines.gears.helical.manufacturing import (pair_backlash, helical_data_sheet, helical_outline, twist_angle)

## Requirements

The rated torque `torque_in` sets the fatigue rating for `life_h` hours. The peak torque `torque_peak`, if given, is checked as a static case: $K_A = 1$, $10^3$ load cycles, and the same minimum safeties. A drive whose duty is mostly well below the peak (vehicles, servo axes, machines with frequent start-ups) should give the **continuous** (or root-mean-cube) torque as `torque_in` and the peak as `torque_peak`.

If `shaft_d2` is left out, the output shaft gets the smallest standard diameter that passes torsion at 30 MPa. The component check then confirms it against bending as well.

In [ ]:
#| export
_SHAFTS = (6, 8, 10, 12, 15, 17, 20, 25, 30, 35, 40, 45, 50, 55, 60)


def standard_shaft(T: float, tau: float = 30.0) -> float:
    "Smallest standard shaft diameter [mm] with torsion stress ≤ tau [MPa] for torque T [N·m]."
    d = (16 * T * 1000 / (math.pi * tau)) ** (1 / 3)
    for s in _SHAFTS:
        if s >= d:
            return float(s)
    return float(math.ceil(d))


@dataclass(frozen=True)
class HelicalRequirements:
    """
    What the stage must do. Lengths mm, torque N·m, speed rpm.

    ratio            : target ratio z2/z1
    torque_in        : pinion torque for the fatigue rating (continuous or rms-cube)
    n_in             : pinion speed
    shaft_d1         : pinion (input) shaft diameter at the gear and the bearings
    shaft_d2         : output shaft diameter (None: sized from torsion, see above)
    torque_peak      : peak pinion torque for the static check (None: not checked)
    life_h           : required life at torque_in and n_in [h]
    priority         : 'quiet', 'balanced' or 'low_thrust' (helix angles tried, see gears.helical.layout)
    centre_distance  : fixed centre distance [mm] (None: free, rounded to a tidy value)
    hub_joint1/2     : pinion / gear joint: 'auto', 'key', 'press', 'adhesive', 'integral'
    ratio_tol        : allowed relative ratio error
    K_A, K_Hbeta     : application factor and face load factor
    materials        : {'pinion'|'gear': key of gears.rating.MATERIALS}
    grade            : ISO 1328-1 flank tolerance class
    flank_finish     : 'ground' or 'hobbed'
    S_Hmin, S_Fmin   : required safety factors (fatigue and peak)
    alternating      : teeth loaded on both flanks (reversing drive), Y_M = 0.7
    m_min, m_max     : module range [mm]
    series_ii        : also allow ISO 54 series II modules
    z1_range         : pinion tooth counts to search
    b_d_max          : largest face width / pinion reference diameter
    face_max         : largest face width [mm] (None: no limit)
    eps_beta_min     : smallest overlap ratio
    prefer_hunting   : rank hunting-tooth pairs ahead of exact ratios within the same 0.5 % band
    span1, span2     : bearing spans [mm] (None: estimated)
    """
    ratio: float
    torque_in: float
    n_in: float
    shaft_d1: float
    shaft_d2: float | None = None
    torque_peak: float | None = None
    life_h: float = 10_000.0
    priority: str = "balanced"
    centre_distance: float | None = None
    hub_joint1: str = "auto"
    hub_joint2: str = "key"
    ratio_tol: float = 0.02
    K_A: float = 1.25
    K_Hbeta: float = 1.15
    materials: dict = field(default_factory=lambda: {"pinion": "case_carburized", "gear": "case_carburized"})
    grade: int = 6
    flank_finish: str = "ground"
    S_Hmin: float = 1.0
    S_Fmin: float = 1.4
    alternating: bool = False
    m_min: float = 0.5
    m_max: float = 4.0
    series_ii: bool = False
    z1_range: tuple[int, int] = (14, 30)
    b_d_max: float = 1.0
    face_max: float | None = None
    eps_beta_min: float = 1.0
    prefer_hunting: bool = False
    span1: float | None = None
    span2: float | None = None

    @property
    def d2(self) -> float:
        "Output shaft diameter [mm] (given, or sized from torsion)."
        return self.shaft_d2 if self.shaft_d2 is not None else standard_shaft(self.K_A * self.torque_in * self.ratio)

    def rating_inputs(self) -> HelicalRatingInputs:
        return HelicalRatingInputs(torque_in=self.torque_in, n_in=self.n_in, life_h=self.life_h, K_A=self.K_A,
                                   K_Hbeta=self.K_Hbeta, grade=self.grade, S_Hmin=self.S_Hmin, S_Fmin=self.S_Fmin,
                                   flank_finish=self.flank_finish)

    def peak_inputs(self) -> HelicalRatingInputs | None:
        if self.torque_peak is None:
            return None
        return HelicalRatingInputs(torque_in=self.torque_peak, n_in=self.n_in, K_A=1.0, K_Hbeta=self.K_Hbeta,
                                   grade=self.grade, S_Hmin=self.S_Hmin, S_Fmin=self.S_Fmin,
                                   flank_finish=self.flank_finish, cycles=1e3)

    def component_inputs(self) -> PairComponentInputs:
        return PairComponentInputs(shaft_d1=self.shaft_d1, shaft_d2=self.d2, hub_joint1=self.hub_joint1,
                                   hub_joint2=self.hub_joint2, span1=self.span1, span2=self.span2,
                                   bearing_life_h=self.life_h)

    def material_set(self) -> dict[str, GearMaterial]:
        return {k: MATERIALS[v] for k, v in self.materials.items()}

## Application Profiles

The same pair can serve very different machines, and what changes is mostly the duty: the required life, the shocks, the noise demands, the accuracy worth paying for, and whether the load reverses. `APPLICATIONS` holds starting values for common uses, and `requirements_for` fills a `HelicalRequirements` from one of them. Any field can still be overridden.

| Profile | Life [h] | $K_A$ | Priority | Class | Reversing | Typical use |
|:--|--:|--:|:--|--:|:--:|:--|
| `industrial` | 20 000 | 1.25 | balanced | 6 | no | general industrial gearbox, electric motor drive, light shocks |
| `uniform` | 40 000 | 1.00 | balanced | 7 | no | pumps, fans, blowers, generators: steady load, long life |
| `moderate_shock` | 20 000 | 1.50 | balanced | 7 | no | conveyors with uneven load, mixers, machine tool main drives |
| `heavy_shock` | 20 000 | 1.75 | low_thrust | 7 | no | crushers, presses, mills, rolling mills |
| `vehicle` | 6 000 | 1.00 | quiet | 5 | yes | EV / e-axle and traction reductions: high speed, NVH-critical, regenerative braking |
| `servo` | 10 000 | 1.25 | balanced | 5 | yes | servo axes, positioning drives, robot joints: reversing, low backlash |
| `tool` | 500 | 1.50 | balanced | 7 | no | power tools, appliances: short life, high speed, hobbed teeth |

$K_A$ follows ISO 6336-6 Annex A for an electric motor or turbine driving a machine with uniform (1.0), light (1.25), moderate (1.5) or heavy (1.75) shocks. The lives are usual design lives at the rated torque (AGMA 6010 and vehicle practice). For a vehicle, it is better to rate a load spectrum (Miner sum) than a single torque. Here the vehicle profile assumes `torque_in` is the damage-equivalent torque, with $K_A = 1$. Each profile is a starting point to be checked against the actual machine.

In [ ]:
#| export
@dataclass(frozen=True)
class ApplicationProfile:
    "Duty-related defaults for one kind of machine (see the table above)."
    description: str
    life_h: float
    K_A: float
    priority: str
    grade: int
    alternating: bool
    flank_finish: str = "ground"


APPLICATIONS: dict[str, ApplicationProfile] = {
    "industrial": ApplicationProfile("general industrial gearbox, electric motor drive, light shocks", 20_000, 1.25, "balanced", 6, False),
    "uniform": ApplicationProfile("pumps, fans, blowers, generators: steady load, long life", 40_000, 1.0, "balanced", 7, False),
    "moderate_shock": ApplicationProfile("conveyors with uneven load, mixers, machine tool main drives", 20_000, 1.5, "balanced", 7, False),
    "heavy_shock": ApplicationProfile("crushers, presses, mills: heavy shocks", 20_000, 1.75, "low_thrust", 7, False),
    "vehicle": ApplicationProfile("EV / e-axle and traction reductions: high speed, NVH-critical, regenerative braking", 6_000, 1.0, "quiet", 5, True),
    "servo": ApplicationProfile("servo axes, positioning drives, robot joints: reversing, low backlash", 10_000, 1.25, "balanced", 5, True),
    "tool": ApplicationProfile("power tools and appliances: short life, high speed", 500, 1.5, "balanced", 7, False, "hobbed"),
}


def requirements_for(application: str, **kw) -> "HelicalRequirements":
    "HelicalRequirements with the defaults of an application profile; keyword arguments override any field."
    if application not in APPLICATIONS:
        raise ValueError(f"application must be one of {list(APPLICATIONS)}")
    a = APPLICATIONS[application]
    base: dict = dict(life_h=a.life_h, K_A=a.K_A, priority=a.priority, grade=a.grade, alternating=a.alternating,
                flank_finish=a.flank_finish)
    base.update(kw)
    return HelicalRequirements(**base)

In [ ]:
#| hide
_r = requirements_for("vehicle", ratio=3.0, torque_in=30.0, n_in=8000.0, shaft_d1=20.0)
assert _r.priority == "quiet" and _r.alternating and _r.K_A == 1.0 and _r.grade == 5
assert requirements_for("industrial", ratio=3.0, torque_in=30.0, n_in=1500.0, shaft_d1=20.0, life_h=5000).life_h == 5000
try:
    requirements_for("boat", ratio=3.0, torque_in=1.0, n_in=1.0, shaft_d1=10.0)
    raise AssertionError
except ValueError:
    pass

## One Candidate

`evaluate` builds the pair, rates it with solid gears to size the hubs, rates it again with the real rims, and sizes the components. It returns `None` if the geometry fails its checks. A candidate **passes** when the fatigue rating, the peak rating (if a peak is given) and every component check pass, and the overlap ratio reaches `eps_beta_min`.

In [ ]:
#| export
@dataclass
class HelicalDesign:
    "A checked design: requirements, pair, ratings, components and backlash allowances."
    req: HelicalRequirements
    pair: HelicalPair
    rating: HelicalPairRating
    rating_peak: HelicalPairRating | None
    components: PairComponents
    allowances: dict

    @property
    def ok(self) -> bool:
        return (self.pair.ok and self.rating.ok and (self.rating_peak is None or self.rating_peak.ok)
                and self.components.ok and self.pair.mesh.eps_beta >= self.req.eps_beta_min - 1e-9)

    @property
    def margin(self) -> float:
        "Smallest ratio of achieved to required safety (fatigue and peak)."
        rs = [self.rating] + ([self.rating_peak] if self.rating_peak else [])
        return min(min(r.S_H / r.S_Hmin, r.S_F / r.S_Fmin) for r in rs)

    @property
    def ratio_error(self) -> float:
        return (self.pair.ratio - self.req.ratio) / self.req.ratio

    def helix_reason(self) -> str:
        "Why this helix angle (for the report)."
        p, mc = self.pair.pinion, self.pair.mesh
        f = self.pair.forces(1.0)
        return (f"Priority '{self.req.priority}' — {HELIX_REASONS[self.req.priority]}. "
                f"β = {math.degrees(p.beta):.2f}° gives ε_β = {mc.eps_beta:.2f} (ε_γ = {mc.eps_gamma:.2f}) "
                f"at b = {p.b:g} mm, and an axial force F_a = {f['F_a'] / f['F_t'] * 100:.0f} % of F_t "
                f"({self.pair.forces(self.req.torque_in)['F_a']:.0f} N at the rated torque).")

    def label(self) -> str:
        p, g = self.pair.pinion, self.pair.gear
        peak = f", peak S_H {self.rating_peak.S_H:.2f} S_F {self.rating_peak.S_F:.2f}" if self.rating_peak else ""
        return (f"{p.z}/{g.z}, m_n {p.mn:g}, β {math.degrees(p.beta):.2f}°, b {p.b:g} mm, a_w {self.pair.a_w:.2f}: "
                f"u = {self.pair.ratio:.4f} ({self.ratio_error * 100:+.2f} %), ε_β {self.pair.mesh.eps_beta:.2f}, "
                f"S_H {self.rating.S_H:.2f}, S_F {self.rating.S_F:.2f}{peak}; limited by {self.rating.limiting}")


def evaluate(req: HelicalRequirements, z1: int, z2: int, mn: float, beta: float, b: float,
             a_w: float | None = None, x_sum: float | None = None) -> HelicalDesign | None:
    "Build, rate and size one candidate; None if its geometry is invalid."
    try:
        if x_sum is not None:
            x1, x2 = split_shift_sum(z1, z2, x_sum, beta)
            pair = HelicalPair.build(z1, z2, mn, beta, b, x1=x1, x2=x2)
        else:
            pair = HelicalPair.build(z1, z2, mn, beta, b, a_w=a_w)
    except ValueError:
        return None
    if not pair.ok:
        return None
    ci = req.component_inputs()
    mats = req.material_set()
    try:
        rims = pair_rims(pair, ci)
        r = rate_helical_pair(pair, req.rating_inputs(), mats, rims, req.alternating)
        pk = req.peak_inputs()
        rp = rate_helical_pair(pair, pk, mats, rims, req.alternating) if pk else None
        c = size_pair_components(pair, r, ci, req.life_h, req.n_in, req.K_A, req.grade)
    except ValueError:
        return None
    return HelicalDesign(req, pair, r, rp, c, pair_backlash(pair, req.grade))

## The Search

For each helix angle of the priority, each tooth count pair within the ratio tolerance, and each standard module from small to large:

1. **Centre distance.** If none is fixed, the tidy value just above the reference centre distance is used. If one is fixed, the shift sum that reaches it must lie within $-0.5 \le x_1 + x_2 \le 1.0$. An extra candidate uses the **exact helix angle** that reaches it with zero shift sum, when that angle lies inside the priority's range.
2. **Face width.** It starts at the width that gives $\varepsilon_\beta$ = `eps_beta_min` (for `'quiet'`, only widths that give a whole-number $\varepsilon_\beta$ = 1, 2, 3) and grows in steps of 1 mm (0.5 mm below $m_n$ = 1) up to `b_d_max`·$d_1$ and `face_max`. The first width that passes is kept.
3. **Next.** Once a module passes for a tooth count pair and angle, larger modules are not tried, because they only make the pair bigger.

Passing designs are ranked by ratio error in 0.5 % bands, then centre distance (in 2 mm bands), then the priority's order of helix angles, then face width, then the larger safety margin.

In [ ]:
#| export
def _faces(req: HelicalRequirements, mn: float, beta: float, d1: float) -> list[float]:
    b_hi = req.b_d_max * d1
    if req.face_max is not None:
        b_hi = min(b_hi, req.face_max)
    b_lo = max(face_width_for_overlap(mn, beta, req.eps_beta_min), 3.0, 4 * mn)
    if req.priority == "quiet":
        return [round(face_width_for_overlap(mn, beta, e) + 0.049, 1) for e in (1, 2, 3)
                if b_lo - 1e-9 <= face_width_for_overlap(mn, beta, e) <= b_hi]
    step = 0.5 if mn < 1 else 1.0
    b0 = math.ceil(b_lo / step - 1e-9) * step
    return [b0 + i * step for i in range(int((b_hi - b0) / step + 1e-9) + 1)]


def design_helical_pair(req: HelicalRequirements, max_results: int = 10) -> list[HelicalDesign]:
    "Search helix angles, tooth counts, modules and face widths; return passing designs, best first (see above)."
    betas = helix_angles(req.priority)
    band = (math.radians(min(betas)), math.radians(max(betas)))
    found: list[tuple[tuple, HelicalDesign]] = []
    counts = pair_tooth_counts(req.ratio, req.ratio_tol, req.z1_range)
    for bi, bdeg in enumerate(betas):
        beta0 = math.radians(bdeg)
        for c in counts:
            for mn in standard_modules(req.m_min, req.m_max, series_ii=req.series_ii):
                trials: list[tuple[float, float | None, float | None]] = []      # (beta, a_w, x_sum)
                if req.centre_distance is None:
                    trials.append((beta0, None, None))
                else:
                    try:
                        xs = helical_shift_sum(c.z1, c.z2, mn, beta0, req.centre_distance)
                        if -0.5 <= xs <= 1.0:
                            trials.append((beta0, req.centre_distance, None))
                    except ValueError:
                        pass
                    if bi == 0:
                        try:
                            be = helix_for_centre_distance(c.z1, c.z2, mn, req.centre_distance)
                            if band[0] <= be <= band[1]:
                                trials.append((be, None, 0.0))
                        except ValueError:
                            pass
                passed = False
                for beta, a_w, xs_ in trials:
                    d1 = c.z1 * mn / math.cos(beta)
                    for b in _faces(req, mn, beta, d1):
                        d = evaluate(req, c.z1, c.z2, mn, beta, b, a_w, xs_)
                        if d is None:
                            break
                        if d.ok:
                            key = (round(abs(d.ratio_error) / 0.005), (not c.hunting) if req.prefer_hunting else 0,
                                   round(d.pair.a_w / 2), bi, b, -d.margin)
                            found.append((key, d))
                            passed = True
                            break
                if passed:
                    break
    found.sort(key=lambda t: t[0])
    return [d for _, d in found[:max_results]]

## Layout Drawing

`layout_svg` draws two views at true scale. The **front view** shows both gears as transverse sections at mean tooth thickness, with their operating pitch circles and the shafts. The **plan section** through both axes shows the gears with their helix lines (opposite hands), the four bearings at their spans, and the located bearing marked "A". A data block lists the main figures.

In [ ]:
#| export
def _poly(P: np.ndarray, sx: float, ox: float, oy: float, style: str) -> str:
    pts = " ".join(f"{ox + x * sx:.2f},{oy - y * sx:.2f}" for x, y in P)
    return f'<polygon points="{pts}" {style}/>'


def layout_svg(d: HelicalDesign, width_px: int = 1100) -> str:
    "SVG text of the front view and plan section of a design (true scale)."
    pr, c = d.pair, d.components
    p, g = pr.pinion, pr.gear
    al = d.allowances["allowances"]
    ra1, ra2 = p.da / 2, g.da / 2
    L1, L2 = c.shafts["pinion"]["span"], c.shafts["gear"]["span"]
    Lmax = max(L1, L2) + 30
    front_w = ra1 + pr.a_w + ra2 + 20
    view_w = front_w + 30 + Lmax + 20
    sx = (width_px - 340) / view_w                   # 340 px kept for the data block
    H = (2 * max(ra1, ra2) + 40) * sx
    H = max(H, (ra1 + pr.a_w + ra2 + 40) * sx)
    ox, oy = (ra1 + 10) * sx, H / 2 + 10
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width_px}" height="{H + 60:.0f}" font-family="sans-serif" font-size="11">',
           '<rect width="100%" height="100%" fill="white"/>']
    # --- front view: pinion at the origin, gear at +a_w ---------------------------------
    ds = {k: (v.A_sne + v.A_sni) / 2 for k, v in al.items()}
    P1 = helical_outline(p, ds["pinion"])
    th = math.pi + math.pi / g.z if (p.z + g.z) % 2 == 0 else math.pi
    R = np.array([[math.cos(th), math.sin(th)], [-math.sin(th), math.cos(th)]])
    P2 = helical_outline(g, ds["gear"]) @ R + [pr.a_w, 0]
    out.append(_poly(P2, sx, ox, oy, 'fill="#f6e7c8" stroke="#222" stroke-width="0.7"'))
    out.append(_poly(P1, sx, ox, oy, 'fill="#d8e8d0" stroke="#222" stroke-width="0.7"'))
    for cx, r_, rs in ((0.0, pr.d_w1 / 2, d.req.shaft_d1 / 2), (pr.a_w, pr.d_w2 / 2, d.req.d2 / 2)):
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy:.2f}" r="{r_ * sx:.2f}" fill="none" stroke="#1f5fa8" stroke-width="0.5" stroke-dasharray="5,3"/>')
        out.append(f'<circle cx="{ox + cx * sx:.2f}" cy="{oy:.2f}" r="{rs * sx:.2f}" fill="#bbb" stroke="#222" stroke-width="0.6"/>')
    out.append(f'<line x1="{ox - (ra1 + 5) * sx:.1f}" y1="{oy:.1f}" x2="{ox + (pr.a_w + ra2 + 5) * sx:.1f}" y2="{oy:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="8" y="{H + 25:.0f}">Front view (transverse section, pitch circles dashed)</text>')
    # --- plan section: axes horizontal, pinion above the gear ------------------------
    x0 = ox + (pr.a_w + ra2 + 30) * sx
    y1 = oy - pr.a_w / 2 * sx
    y2 = oy + pr.a_w / 2 * sx

    def rect(x, y_c, length, half_h, fill, extra=""):
        return (f'<rect x="{x:.2f}" y="{y_c - half_h * sx:.2f}" width="{length * sx:.2f}" height="{2 * half_h * sx:.2f}" '
                f'fill="{fill}" stroke="#222" stroke-width="0.6" {extra}/>')
    for key, gg, yc, L, dsh in (("gear", g, y2, L2, d.req.d2), ("pinion", p, y1, L1, d.req.shaft_d1)):
        xs = x0 + 15 * sx
        out.append(rect(x0, yc, Lmax, dsh / 2, "#ddd"))                                           # shaft
        xg = xs + c.shafts[key]["gear_from_A"] * sx - gg.b / 2 * sx
        out.append(rect(xg, yc, gg.b, gg.da / 2, "#d8e8d0" if key == "pinion" else "#f6e7c8"))
        # helix lines on the visible face (slope tan β; hands opposite)
        n_l = 6
        sgn = 1 if gg.hand == "right" else -1
        h = gg.da / 2
        for i in range(n_l + 1):
            u = -h + 2 * h * i / n_l
            dx = gg.b
            dy = sgn * dx * math.tan(gg.beta)
            ya, yb = u, u + dy
            if abs(dy) > 1e-9:
                # clip to the gear rectangle
                pts = []
                for t in np.linspace(0, 1, 50):
                    yy = ya + (yb - ya) * t
                    if -h <= yy <= h:
                        pts.append((xg + t * dx * sx, yc - yy * sx))
                if len(pts) > 1:
                    out.append(f'<line x1="{pts[0][0]:.2f}" y1="{pts[0][1]:.2f}" x2="{pts[-1][0]:.2f}" y2="{pts[-1][1]:.2f}" stroke="#555" stroke-width="0.5"/>')
        for side, xb in (("A", xs), ("B", xs + L * sx)):
            info = c.bearings[f"{key}_{side}"]
            D_o, B_w, d_b = info["D"], info["width"], info["bore"]
            out.append(rect(xb - B_w / 2 * sx, yc - (d_b / 2 + (D_o - d_b) / 4) * sx, B_w, (D_o - d_b) / 4, "#999"))
            out.append(rect(xb - B_w / 2 * sx, yc + (d_b / 2 + (D_o - d_b) / 4) * sx, B_w, (D_o - d_b) / 4, "#999"))
            out.append(f'<text x="{xb:.1f}" y="{yc - (D_o / 2 + 2) * sx:.1f}" text-anchor="middle" font-size="9">{side}</text>')
        out.append(f'<line x1="{x0 - 5 * sx:.1f}" y1="{yc:.1f}" x2="{x0 + (Lmax + 5) * sx:.1f}" y2="{yc:.1f}" stroke="#a33" stroke-width="0.4" stroke-dasharray="8,2,2,2"/>')
    out.append(f'<text x="{x0 + Lmax / 2 * sx:.1f}" y="{H + 25:.0f}" text-anchor="middle">Plan section (A = located bearing)</text>')
    # --- data block ------------------------------------------------------------------
    tx = x0 + (Lmax + 15) * sx
    f = pr.forces(d.req.torque_in)
    lines = [f"u = {pr.ratio:.4f} ({p.z}/{g.z})",
             f"m_n {p.mn:g} mm, α_n 20°, β {math.degrees(p.beta):.2f}° ({p.hand}/{g.hand})",
             f"a_w = {pr.a_w:.3f} mm, b = {p.b:g} / {g.b:g} mm",
             f"ε_α {pr.mesh.eps_alpha:.2f}, ε_β {pr.mesh.eps_beta:.2f}, ε_γ {pr.mesh.eps_gamma:.2f}",
             f"T_in {d.req.torque_in:g} N·m at {d.req.n_in:g} rpm" + (f", peak {d.req.torque_peak:g}" if d.req.torque_peak else ""),
             f"F_t {f['F_t']:.0f} N, F_r {f['F_r']:.0f} N, F_a {f['F_a']:.0f} N",
             f"S_H {d.rating.S_H:.2f} / S_F {d.rating.S_F:.2f}" + (f" (peak {d.rating_peak.S_H:.2f} / {d.rating_peak.S_F:.2f})" if d.rating_peak else ""),
             f"pinion: {c.pinion_hub['joint']} on Ø{d.req.shaft_d1:g}",
             f"gear: {c.gear_hub['joint']} on Ø{d.req.d2:g}",
             f"bearings A: {c.bearings['pinion_A']['bearing'].split(' (')[0]} / {c.bearings['gear_A']['bearing'].split(' (')[0]}",
             f"bearings B: {c.bearings['pinion_B']['bearing'].split(' (')[0]} / {c.bearings['gear_B']['bearing'].split(' (')[0]}"]
    for i, s in enumerate(lines):
        out.append(f'<text x="{tx:.1f}" y="{30 + 16 * i}">{s}</text>')
    out.append("</svg>")
    return "\n".join(out)

## Package

| File | Content |
|:--|:--|
| `report.md` | requirements, helix angle reasoning, gear pair, forces, fatigue and peak rating, backlash, gear data, components, checks, parts list, assumptions |
| `gear_data_pinion.csv`, `gear_data_gear.csv` | ISO 1340 data tables for the gear drawings |
| `parts.csv` | parts list |
| `component_dimensions.csv` | fits and limits for the part drawings |
| `layout.svg` | front view and plan section |
| `pinion.dxf`, `gear.dxf` | transverse outlines at mean thickness (twist angle in the report) |

In [ ]:
#| export
def _fmt(v) -> str:
    return f"{v:.4g}" if isinstance(v, float) else str(v)


def _dev(dn: float, fit: str) -> str:
    up, lo = iso286_limits(dn, fit)
    f = lambda v: "0" if abs(v) < 5e-7 else f"{v:+.3f}"
    return f"({f(up)}/{f(lo)})"


def data_sheets(d: HelicalDesign) -> dict[str, list[DataRow]]:
    "Gear data tables of pinion and gear."
    pr, al = d.pair, d.allowances["allowances"]
    mats = d.req.material_set()
    fin = d.req.flank_finish
    integ = d.components.pinion_hub["joint"] == "integral"
    return {"pinion": helical_data_sheet(pr.pinion, pr.gear, pr.a_w, al["pinion"], d.req.grade, mats["pinion"], fin,
                                         pr.mesh.d_Nf1, integral=integ),
            "gear": helical_data_sheet(pr.gear, pr.pinion, pr.a_w, al["gear"], d.req.grade, mats["gear"], fin,
                                       pr.mesh.d_Nf2, integral=d.components.gear_hub["joint"] == "integral")}


def component_rows(d: HelicalDesign) -> list[tuple[str, str, str]]:
    "(part, feature, specification) rows with numeric limits for the part drawings."
    c, pr = d.components, d.pair
    rows: list[tuple[str, str, str]] = []
    for key, hub, gg, dsh in (("pinion", c.pinion_hub, pr.pinion, d.req.shaft_d1), ("gear", c.gear_hub, pr.gear, d.req.d2)):
        shaft = "input shaft" if key == "pinion" else "output shaft"
        if hub["joint"] == "integral":
            rows.append((f"{key} shaft", "teeth", f"cut on the shaft; {hub['note']}"))
        elif hub["joint"] == "key":
            rows += [(key, "bore", fmt_fit(dsh, "H7") + f" (shaft k6), hub length {hub['hub_length']:g} mm"),
                     (key, "key groove", f"width {hub['key_b']:g} P9 {_dev(hub['key_b'], 'P9')}, depth {dsh + hub['t2']:.1f} +0.1 over bore side"),
                     (shaft, "key seat", f"width {hub['key_b']:g} P9, depth t1 = {hub['t1']:g} +0.1 mm; {hub['key']}"),
                     (shaft, "axial location", hub["axial"])]
        elif hub["joint"] == "press":
            h_, s_ = hub["press_fit"].split("/")
            rows += [(key, "bore", fmt_fit(dsh, h_) + f", Rz ≤ 2 µm; interference fit {hub['press_fit']}"),
                     (shaft, "gear seat", fmt_fit(dsh, s_) + f", Rz ≤ 2 µm; shrink at {hub['shrink_temperature']:.0f} °C")]
        elif hub["joint"] == "adhesive":
            rows += [(key, "bore", fmt_fit(dsh, "H7") + ", bonded with anaerobic retaining compound"),
                     (shaft, "gear seat", fmt_fit(dsh, "k6") + ", degreased and activated")]
        for side in ("A", "B"):
            b = c.bearings[f"{key}_{side}"]
            rows.append((shaft, f"bearing journal {side}", fmt_fit(b["bore"], "k5") + f" for {b['bearing'].split(' (')[0]} "
                         f"({b['role']}), shoulder per bearing maker"
                         + (f"; journal stepped up from Ø{dsh:g}" if b["bore"] > dsh + 1e-9 else "")))
        rows.append((gg.name, "faces", f"run-out to bore/journals ≤ {max(0.005, it_grade(gg.d, 6)):.3f} mm"))
    h = c.housing
    rows += [("housing", "centre distance", h["centre_distance"]),
             ("housing", "axis parallelism", f"f_Σβ ≤ {h['f_Sigma_beta'] * 1000:.0f} µm, f_Σδ ≤ {h['f_Sigma_delta'] * 1000:.0f} µm "
                                             f"over the {h['bearing_span']:.0f} mm bearing span (ISO/TR 10064-3)"),
             ("housing", "bearing bores", h["bores"])]
    return rows


def parts_list(d: HelicalDesign) -> list[tuple]:
    "(item, part, qty, description, material/standard)."
    pr, c = d.pair, d.components
    mats = d.req.material_set()
    p, g = pr.pinion, pr.gear
    items = []
    if c.pinion_hub["joint"] == "integral":
        items.append(("Pinion shaft", 1, f"z {p.z}, m_n {p.mn:g}, β {math.degrees(p.beta):.2f}° {p.hand}, b {p.b:g}, journals Ø{d.req.shaft_d1:g} k5",
                      mats["pinion"].name + ", " + mats["pinion"].treatment))
    else:
        items.append(("Pinion", 1, f"z {p.z}, m_n {p.mn:g}, β {math.degrees(p.beta):.2f}° {p.hand}, b {p.b:g}, bore Ø{d.req.shaft_d1:g}",
                      mats["pinion"].name + ", " + mats["pinion"].treatment))
    items.append(("Gear", 1, f"z {g.z}, m_n {g.mn:g}, β {math.degrees(g.beta):.2f}° {g.hand}, b {g.b:g}, bore Ø{d.req.d2:g}"
                  if c.gear_hub["joint"] != "integral" else f"z {g.z}, cut on the output shaft",
                  mats["gear"].name + ", " + mats["gear"].treatment))
    for key, hub, dsh in (("pinion", c.pinion_hub, d.req.shaft_d1), ("gear", c.gear_hub, d.req.d2)):
        if hub["joint"] == "key":
            items.append((f"Parallel key ({key})", 1, hub["key"], "C45+C, DIN 6885-1"))
            items.append((f"Retaining ring ({key})", 1, f"DIN 471 for Ø{dsh:g} shaft (axial location against F_a)", "spring steel"))
    if c.gear_hub["joint"] != "integral":
        items.append(("Output shaft", 1, f"Ø{d.req.d2:g}, journals k5", "42CrMo4 Q&T (1.7225) or C45"))
    by_type: dict[str, int] = {}
    for k_, b in c.bearings.items():
        by_type[b["bearing"]] = by_type.get(b["bearing"], 0) + 1
    for bt, n in by_type.items():
        items.append(("Deep groove ball bearing", n, bt, "ISO 15, 2RS or Z shields as the lubrication needs"))
    items.append(("Bearing retaining ring / cover", 2, "locate bearing A of each shaft in both directions", ""))
    items.append(("Shaft seal", 2, f"radial lip seals Ø{d.req.shaft_d1:g} and Ø{d.req.d2:g} (DIN 3760) where shafts leave the housing", "NBR or FKM"))
    return [(i + 1, *row) for i, row in enumerate(items)]


def report_markdown(d: HelicalDesign) -> str:
    "The design report (see the package table above)."
    r, pr, c, req = d.rating, d.pair, d.components, d.req
    p, g = pr.pinion, pr.gear
    f = pr.forces(req.torque_in)
    sp = pr.speeds(req.n_in)
    L = ["# Single-stage helical gear pair: design report", "",
         "External helical pair, pinion driving. 20° normal pressure angle, ISO 53 profile A. "
         "Lengths in mm, torques in N·m, stresses in MPa. Calculated with `emachines.gears.helical`.", "",
         "## Requirements", "", "| Item | Value |", "|---|---|"]
    for k, v in asdict(req).items():
        L.append(f"| {k} | {v} |")
    if req.shaft_d2 is None:
        L.append(f"| output shaft (sized) | Ø{req.d2:g} |")
    L += ["", "## Helix angle", "", d.helix_reason(), ""]
    if math.gcd(p.z, g.z) != 1:
        L += [f"The tooth counts {p.z}/{g.z} share the factor {math.gcd(p.z, g.z)} (not hunting), because the exact ratio "
              f"{req.ratio:g} was preferred. That is acceptable for hardened and ground gears. For hobbed or through-hardened "
              "gears, a hunting pair within the ratio tolerance (`prefer_hunting=True`) spreads wear better.", ""]
    L += ["## Gear pair", "", "```", pr.summary(), "```", "",
          f"Speeds: pinion {sp['pinion']:g} rpm, gear {sp['gear']:.1f} rpm (opposite direction). "
          f"Mesh frequency {pr.mesh_frequency(req.n_in):.0f} Hz. Pitch line velocity {pr.pitch_line_velocity(req.n_in):.2f} m/s.", "",
          f"Tooth forces at the rated torque {req.torque_in:g} N·m: F_t = {f['F_t']:.0f} N, F_r = {f['F_r']:.0f} N, "
          f"F_a = {f['F_a']:.0f} N (on the operating pitch circle Ø{pr.d_w1:.3f})."
          + (f" At the peak {req.torque_peak:g} N·m: F_t = {pr.forces(req.torque_peak)['F_t']:.0f} N, "
             f"F_a = {pr.forces(req.torque_peak)['F_a']:.0f} N." if req.torque_peak else ""), "",
          f"CAD: extrude the DXF outline by the face width with a twist of {twist_angle(p):+.2f}° (pinion) and "
          f"{twist_angle(g):+.2f}° (gear); positive is a right-hand helix.", "",
          "## Load capacity (ISO 6336)", "", f"### Fatigue at {req.torque_in:g} N·m, {req.n_in:g} rpm, {req.life_h:g} h", "",
          "Factors: " + ", ".join(f"{k} = {v:.3f}" for k, v in r.factors.items()), "", "```", r.table(), "```", ""]
    if d.rating_peak:
        L += [f"### Static check at the peak {req.torque_peak:g} N·m (K_A = 1, 10³ cycles)", "", "```", d.rating_peak.table(), "```", ""]
    al = d.allowances
    L += ["## Backlash and tooth thickness", "",
          f"Required minimum normal backlash {al['j_bn_min_required']:.3f} mm; centre distance tolerance ±{al['f_a'] * 1000:.0f} µm.", "",
          "| Gear | A_sne | A_sni |", "|---|---|---|"]
    for k, a in al["allowances"].items():
        L.append(f"| {k} | {a.A_sne:+.3f} | {a.A_sni:+.3f} |")
    L += ["", f"Normal backlash {al['backlash_n'][0]:.3f} to {al['backlash_n'][1]:.3f} mm; circumferential on the pitch "
              f"circle {al['backlash_t'][0]:.3f} to {al['backlash_t'][1]:.3f} mm "
              f"({math.degrees(al['backlash_t'][0] / (pr.d_w2 / 2)) * 60:.1f}′ to {math.degrees(al['backlash_t'][1] / (pr.d_w2 / 2)) * 60:.1f}′ at the output)."]
    for name, rows in data_sheets(d).items():
        L += ["", f"## Gear data — {name}", "", "| Item | Symbol | Value |", "|---|---|---|"]
        L += [f"| {x.item} | {x.symbol} | {x.value} |" for x in rows]
    L += ["", "## Components", "", "| Part | Feature | Specification |", "|---|---|---|"]
    L += [f"| {a} | {b_} | {s} |" for a, b_, s in component_rows(d)]
    L += ["", "### Pinion joint", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.pinion_hub.items()]
    L += ["", "### Gear joint", ""] + [f"- {k}: {_fmt(v)}" for k, v in c.gear_hub.items()]
    for k, s in c.shafts.items():
        L += ["", f"### {k.capitalize()} shaft", ""] + [f"- {kk}: {_fmt(v)}" for kk, v in s.items()]
    L += ["", "### Bearings", "", "| Bearing | Role | F_R [N] | F_a [N] | P [N] | n [rpm] | C_req [kN] | Selected | L10h ≈ [h] |",
          "|---|---|---|---|---|---|---|---|---|"]
    for k, b in c.bearings.items():
        L.append(f"| {k} | {b['role']} | {b['F_R']:.0f} | {b['F_a']:.0f} | {b['P']:.0f} | {b['speed_rpm']:.0f} | "
                 f"{b['C_req'] / 1000:.2f} | {b['bearing']} | {b['L10h_est']:.3g} |")
    L += ["", "## Checks", ""] + [f"- {ch}" for ch in c.checks]
    L += [f"- {'✓' if r.ok else '✗'} fatigue rating: S_H {r.S_H:.2f} ≥ {req.S_Hmin}, S_F {r.S_F:.2f} ≥ {req.S_Fmin}"]
    if d.rating_peak:
        L += [f"- {'✓' if d.rating_peak.ok else '✗'} peak (static): S_H {d.rating_peak.S_H:.2f} ≥ {req.S_Hmin}, "
              f"S_F {d.rating_peak.S_F:.2f} ≥ {req.S_Fmin}"]
    L += [f"- {'✓' if pr.mesh.eps_beta >= req.eps_beta_min - 1e-9 else '✗'} overlap ratio ε_β {pr.mesh.eps_beta:.2f} ≥ {req.eps_beta_min}"]
    L += [f"- note: {w}" for w in pr.warnings()]
    for k, s in c.shafts.items():
        L.append(f"- note: {k} shaft deflection at the mesh {s['deflection_at_gear'] * 1000:.1f} µm "
                 f"({s['deflection_per_mn']:.3f} m_n; keep below about 0.01 m_n or raise K_Hβ)")
    L += ["", "## Lubrication", "", c.lubrication, "",
          "## Parts list", "", "| # | Part | Qty | Description | Material / standard |", "|---|---|---|---|---|"]
    L += [f"| {i} | {pt} | {q} | {ds} | {mt} |" for i, pt, q, ds, mt in parts_list(d)]
    L += ["", "## Assumptions to confirm before release", "",
          "- Material allowables are ISO 6336-5 MQ typical values; confirm with the steel certificate and heat treater.",
          f"- K_Hβ = {req.K_Hbeta} assumes the gear between two bearings in a stiff housing, with a crowned or helix-corrected pinion.",
          "- K_Hα uses the ISO 6336-1 table; below K_A·F_t/b = 100 N/mm it takes the light-load value ε_αn.",
          "- Bearing ratings are typical catalogue values for deep groove ball bearings; confirm C_r, C_0 and the speed limit "
          "with the bearing maker, and consider an angular contact or four-point bearing at A if F_a/F_R is high.",
          "- Bearing loads take the unfavourable sign of the axial moment on both bearings (reversing drive).",
          "- Tooth thickness tolerances follow DIN 3967 practice; agree them with the gear shop.",
          "- The DXF fillet is a circle approximating the hob's elliptical transverse fillet.",
          "- Re-check the final design in a full ISO 6336 tool if any safety factor is within 10 % of its minimum."]
    return "\n".join(L) + "\n"


def write_package(d: HelicalDesign, folder: str) -> list[str]:
    "Write the manufacturing package (see the table above) into `folder`; returns the file paths."
    os.makedirs(folder, exist_ok=True)
    paths: list[str] = []

    def pth(name):
        paths.append(os.path.join(folder, name))
        return paths[-1]
    with open(pth("report.md"), "w") as fh:
        fh.write(report_markdown(d))
    for name, rows in data_sheets(d).items():
        with open(pth(f"gear_data_{name}.csv"), "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(["item", "symbol", "value"])
            w.writerows([(x.item, x.symbol, x.value) for x in rows])
    with open(pth("parts.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["item", "part", "qty", "description", "material_or_standard"])
        w.writerows(parts_list(d))
    with open(pth("component_dimensions.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["part", "feature", "specification"])
        w.writerows(component_rows(d))
    with open(pth("layout.svg"), "w") as fh:
        fh.write(layout_svg(d))
    al = d.allowances["allowances"]
    pr = d.pair
    for key, gg, dsh, hub in (("pinion", pr.pinion, d.req.shaft_d1, d.components.pinion_hub),
                              ("gear", pr.gear, d.req.d2, d.components.gear_hub)):
        circles = [] if hub["joint"] == "integral" else [(0, 0, dsh / 2)]
        write_dxf(pth(f"{key}.dxf"), [helical_outline(gg, (al[key].A_sne + al[key].A_sni) / 2)], circles)
    return paths


HelicalDesign.report = report_markdown            # type: ignore[attr-defined]
HelicalDesign.write_package = write_package       # type: ignore[attr-defined]
HelicalDesign.svg = layout_svg                    # type: ignore[attr-defined]

## Example: Ratio 3, 30–40 N·m Peak

Ratio 3 with 30–40 N·m peak on the pinion and a 15 mm input shaft, as a general-purpose industrial stage. The fatigue rating uses 20 N·m continuous at 3000 rpm for the profile's 20,000 h, and the static check uses the 40 N·m peak. The helix angle is left to the profile's priority.

In [ ]:
req = requirements_for("industrial", ratio=3.0, torque_in=20.0, torque_peak=40.0, n_in=3000.0, shaft_d1=15.0)
designs = design_helical_pair(req, max_results=5)
for d in designs:
    print(d.label())
best = designs[0]
print()
print(best.helix_reason())
print(best.pair.summary())

In [ ]:
#| hide
assert designs and all(d.ok for d in designs)
assert abs(best.ratio_error) <= req.ratio_tol
assert best.rating_peak is not None and best.rating_peak.ok
assert best.pair.mesh.eps_beta >= 1 - 1e-9
assert round(math.degrees(best.pair.pinion.beta), 6) in HELIX_PRIORITIES["balanced"]
_svg = layout_svg(best)
assert _svg.startswith("<svg") and _svg.rstrip().endswith("</svg>")
_md = report_markdown(best)
for _s in ("## Helix angle", "## Load capacity", "## Gear data — pinion", "### Bearings", "## Parts list"):
    assert _s in _md, _s
import tempfile
with tempfile.TemporaryDirectory() as _tmp:
    _files = write_package(best, _tmp)
    assert len(_files) == 8 and all(os.path.getsize(f_) > 100 for f_ in _files)

In [ ]:
#| hide
# a fixed centre distance is met exactly, and the low-thrust priority gives smaller helix angles
_req = HelicalRequirements(ratio=3.0, torque_in=20.0, n_in=3000.0, life_h=2000, shaft_d1=15.0,
                           centre_distance=60.0, priority="low_thrust")
_ds = design_helical_pair(_req, max_results=3)
assert _ds and all(abs(d.pair.a_w - 60.0) < 1e-6 for d in _ds)
assert all(math.degrees(d.pair.pinion.beta) <= 12 + 1e-6 for d in _ds)
# quiet priority: whole-number overlap
_q = design_helical_pair(HelicalRequirements(ratio=3.0, torque_in=20.0, n_in=3000.0, life_h=2000, shaft_d1=15.0,
                                             priority="quiet"), max_results=2)
assert _q and all(abs(d.pair.mesh.eps_beta - round(d.pair.mesh.eps_beta)) < 0.03 for d in _q)

### The same stage for different machines

The same ratio and torques, designed for each application profile at a speed typical for it. Life, shocks, accuracy and reversing change the module, the face width and the helix angle that come out.

In [ ]:
speeds = {"industrial": 1500, "uniform": 1500, "moderate_shock": 1500, "heavy_shock": 1000, "vehicle": 8000,
          "servo": 3000, "tool": 12000}
print(f"{'profile':15s} {'n_in':>6s}  design")
for app, n in speeds.items():
    r_ = requirements_for(app, ratio=3.0, torque_in=20.0, torque_peak=40.0, n_in=float(n), shaft_d1=15.0)
    ds_ = design_helical_pair(r_, max_results=1)
    print(f"{app:15s} {n:6d}  " + (ds_[0].label() if ds_ else "no design in the search range"))

The layout drawing of the best design:

In [ ]:
#| echo: false
from IPython.display import SVG, display
display(SVG(layout_svg(best)))

## References

1. ISO 6336-1, -2, -3, -5 (rating, via `gears.helical.rating`).
2. ISO 21771:2007 (geometry), ISO 1328-1:2013 (tolerances), ISO 1340 / DIN 3966-1 (data tables).
3. ISO 281:2007 (bearing life), ISO/TR 10064-3:1996 (housing and axis tolerances).
4. G. Niemann, H. Winter, *Maschinenelemente*, Band II, Springer, 2nd ed., 1989 (helical gear design practice).